In [1]:
import brightway2 as bw
from premise import *
import os
import wurst
import time
import copy

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v3') # set current project

In [5]:
from premise_gwp import add_premise_gwp
add_premise_gwp()

Adding ('IPCC 2013', 'climate change', 'GWP 100a, incl. H')
Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.10 seconds
Wrote 1 LCIA methods with 216 characterization factors
Adding ('IPCC 2013', 'climate change', 'GTP 20a, incl. bio CO2')
Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.05 seconds
Wrote 1 LCIA methods with 21

In [6]:
bioSphereDBName = 'biosphere3'
bioSphereDB = bw.Database(bioSphereDBName) # import the biosphere database
ecoInventBase2020DBName = 'ecoinvent 3.8 cutoff image SSP2-Base 2020'
ecoInventBase2020DB = bw.Database(ecoInventBase2020DBName) # import the ecoinvent database (image base 2020)

In [7]:
# import base inventories from excel
excelImport = bw.ExcelImporter(os.path.join('..', 'Raw data', 'lci-Abhi image SSP2-Base 2020 latest.xlsx'))
excelImport.apply_strategies()
excelImport.match_database(ecoInventBase2020DBName, fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excelImport.match_database(bioSphereDBName, fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excelImport.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from my database
excelImport.statistics()

Extracted 4 worksheets in 0.03 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.07 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
24 datasets
256 exchanges
0 unlinked exchang

(24, 256, 0)

In [8]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excelImport:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [9]:
lciBase2020DBName = 'lci-Abhi image SSP2-Base 2020'

In [9]:
newLocations = {'BR' : 'Brazil',
                'CA' : 'Canada',
                'PL' : 'Central Europe',
                'CN' : 'China',
                'ET' : 'Eastern Africa',
                'IN' : 'India',
                'ID' : 'Indonesia',
                'JP' : 'Japan',
                'KR' : 'Korea',
                'IR' : 'Middle East',
                'MX' : 'Mexico',
                'EG' : 'Northern Africa',
                'AU' : 'Oceania',
                'GT' : 'Rest of Central America',
                'BW' : 'Rest of Southern Africa',
                'CL' : 'Rest of Southern America',
                'PK' : 'Rest of Southern Asia',
                'RU' : 'Russia',
                'ZA' : 'South Africa',
                'PH' : 'South Eastern Asia',
                'UZ' : 'Central Asia',
                'TR' : 'Turkey',
                'UA' : 'Ukraine',
                'US' : 'United States of America',
                'NG' : 'Western Africa',
                'RER' : 'Western Europe'}

In [11]:
globalDB = excelImport.data # add all inventories
ecoInventBase2020DBDict = [ds.as_dict() for ds in ecoInventBase2020DB] # convert ecoinvent database to dictionary
bioSphereDBDict = [ds.as_dict() for ds in bioSphereDB] # convert biosphere database to dictionary # maybe not needed?

DSToRegionalize = globalDB

regionalizedDS = []
for ds in DSToRegionalize:
    for loc in newLocations:
        dsCopy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalizedDS.append(dsCopy)        

# Relink technosphere exchanges to the new locations
for ds in regionalizedDS:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoInventBase2020DBName:
                dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
                if not dsOutput and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
            elif exchange['database'] == lciBase2020DBName:
                dsOutput = [dsInstance for dsInstance in regionalizedDS if exchange['name'] == dsInstance['name']
                                and ds['location'] == dsInstance['location']]
            if dsOutput:
                    exchange.update({'location' : dsOutput[0]['location']})

In [12]:
db = globalDB + regionalizedDS
DBLinked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in DBLinked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input' : (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchangeLink = wurst.get_one(db + ecoInventBase2020DBDict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchangeLink = [ds['code'] for ds in bioSphereDBDict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input' : (bioSphereDBName, exchangeLink)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [13]:
if lciBase2020DBName in bw.databases:
    del bw.databases[lciBase2020DBName]  
wurst.write_brightway2_database(DBLinked, lciBase2020DBName) # write database

648 datasets
6912 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 08/11/2023 18:38:53
  Finished: 08/11/2023 18:38:53
  Total time elapsed: 00:00:00
  CPU %: 96.10
  Memory %: 1.34
Created database: lci-Abhi image SSP2-Base 2020


In [14]:
lciBase2020DB = wurst.extract_brightway2_databases(lciBase2020DBName)

Getting activity data


100%|██████████| 648/648 [00:00<00:00, 328766.06it/s]


Adding exchange data to activities


100%|██████████| 6912/6912 [00:00<00:00, 89605.43it/s]


Filling out exchange data


100%|██████████| 648/648 [00:00<00:00, 8399.96it/s]


In [15]:
databaseNames = bw.databases
ecoInventDBNames = []
premiseDBNames = []
for databaseName in databaseNames:
    if '2020' not in databaseName and 'SSP2' in databaseName:
        ecoInventDBNames.append(databaseName)
        premiseDBNames.append(databaseName.replace('ecoinvent 3.8 cutoff ', ''))
ecoInventDBNames.sort()
premiseDBNames.sort()

In [16]:
for premiseDBName in premiseDBNames:
    print('linking database ' + premiseDBName + '...')
    premiseDBDict = [ds.as_dict() for ds in bw.Database('ecoinvent 3.8 cutoff ' + premiseDBName)]

    DBLinked = copy.deepcopy(lciBase2020DB)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in DBLinked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchangeLink = wurst.get_one(lciBase2020DB + premiseDBDict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
                exchange.update({'database' : exchangeLink['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchangeLink = [ds['code'] for ef in bioSphereDBDict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchangeLink)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    dbName = 'lci-Abhi ' + premiseDBName
    if dbName in bw.databases:
        del bw.databases[dbName]
    wurst.write_brightway2_database(DBLinked, dbName)
    print(premiseDBName + ' linking and writing successful!')

linking database image SSP2-Base 2030...
648 datasets
6912 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 08/11/2023 18:40:26
  Finished: 08/11/2023 18:40:26
  Total time elapsed: 00:00:00
  CPU %: 97.00
  Memory %: 4.58
Created database: lci-Abhi image SSP2-Base 2030
image SSP2-Base 2030 linking and writing successful!
linking database image SSP2-Base 2040...
648 datasets
6912 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 08/11/2023 18:42:02
  Finished: 08/11/2023 18:42:02
  Total time elapsed: 00:00:00
  CPU %: 96.10
  Memory %: 1.72
Created database: lci-Abhi image SSP2-Base 2040
image SSP2-Base 2040 linking and writing successful!
linking database image SSP2-Base 2050...
648 datasets
6912 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 08/11/2023 18:43:45
  Finished: 08/11/2023 18:43:45
  Total time elapsed: 00:00:00
  CPU %: 96.10
  Memory %: 5.79


In [4]:
allDBNames = list(bw.databases)

In [5]:
myDBNames = []
for DBName in allDBNames:
    if 'Abhi' in DBName:
        myDBNames.append(DBName)

In [6]:
ecoinventSSP2DBNames = []
for DBName in allDBNames:
    if 'SSP2' in DBName and 'ecoinvent' in DBName:
        ecoinventSSP2DBNames.append(DBName)
ecoinventSSP2DBNames.sort()

In [7]:
imageLocations = {'Global' : 'World',
                  'Brazil' : 'BRA',
                  'Canada' : 'CAN',
                  'Central Europe' : 'CEU',
                  'China' : 'CHN',
                  'Eastern Africa' : 'EAF',
                  'India' : 'INDIA',
                  'Indonesia' : 'INDO',
                  'Japan' : 'JAP',
                  'Korea' : 'KOR',
                  'Middle East' : 'ME',
                  'Mexico' : 'MEX',
                  'Northern Africa' : 'NAF',
                  'Oceania' : 'OCE',
                  'Rest of Central America' : 'RCAM',
                  'Rest of Southern Africa' : 'RSAF',
                  'Rest of Southern America' : 'RSAM',
                  'Rest of Southern Asia' : 'RSAS',
                  'Russia' : 'RUS',
                  'South Africa' : 'SAF',
                  'South Eastern Asia' : 'SEAS',
                  'Central Asia' : 'STAN',
                  'Turkey' : 'TUR',
                  'Ukraine' : 'UKR',
                  'United States of America' : 'USA',
                  'Western Africa' : 'WAF',
                  'Western Europe' : 'WEU'}

def find_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

In [10]:
# generate all hydrogen, carbon dioxide, and ammonia inventories
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database(ecoinventSSP2DBName)
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database(mySSP2DBName)
    print('Started ' + mySSP2DBName)

    locations = imageLocations

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
    
        if ecoinventLocation:
            newLoc = ecoinventLocation
            
            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not solarElecAct:
                solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not onshoreWindElecAct:
                onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            offshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, 1-3MW turbine, offshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not offshoreWindElecAct:
                offshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, 1-3MW turbine, offshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            nuclearElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, nuclear, pressure water reactor' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not nuclearElecAct:
                nuclearElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, nuclear, pressure water reactor' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            hydroElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, hydro, run-of-river' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not hydroElecAct:
                hydroElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, hydro, run-of-river' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
                
            geothermalElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, deep geothermal' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and ecoinventLocation == activity['location']]
            if not geothermalElecAct:
                geothermalElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, deep geothermal' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
                
        else:
            newLoc = 'GLO'

            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            offshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, 1-3MW turbine, offshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            nuclearElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, nuclear, pressure water reactor' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            hydroElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, hydro, run-of-river' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            geothermalElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, deep geothermal' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
        
        # hydrogen inventories
        hydrogenBAUAct = [activity for activity in ecoinventSSP2DB 
                    if 'hydrogen production, steam methane reforming of natural gas, 25 bar' in activity['name']
                    and location == activity['location']][0]
        hydrogenBAUActCopy = hydrogenBAUAct.copy(database = mySSP2DB.name, location = newLoc)
        hydrogenBAUActCopy.update({'reference product' : 'hydrogen, Abhi'})
        hydrogenBAUActCopy.save()
        
        hydrogenBlueAct = [activity for activity in ecoinventSSP2DB 
                       if r'hydrogen production, steam methane reforming of natural gas, with CCS (MDEA, 98% eff.), 25 bar' in activity['name']
                       and location == activity['location']][0]
        hydrogenBlueActCopy = hydrogenBlueAct.copy(database = mySSP2DB.name, location = newLoc)
        hydrogenBlueActCopy.update({'reference product' : 'hydrogen, Abhi'})
        hydrogenBlueActCopy.save()

        hydrogenPEMGridAct = [activity for activity in ecoinventSSP2DB 
                          if 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from grid electricity' in activity['name']
                          and location == activity['location']][0]
        hydrogenPEMGridActCopy = hydrogenPEMGridAct.copy(database = mySSP2DB.name, location = newLoc)
        hydrogenPEMGridActCopy.update({'reference product' : 'hydrogen, Abhi'})
        hydrogenPEMGridActCopy.save()

        hydrogenPEMSolarCopy = hydrogenPEMGridActCopy.copy(name = 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from solar electricity', location = newLoc)
        elecHydrogenPEMSolarCopy = [exchange for exchange in hydrogenPEMSolarCopy.technosphere()
                                    if 'electricity, low voltage' in exchange['product']][0]
        elecHydrogenPEMSolarCopy.input = solarElecAct[0]
        elecHydrogenPEMSolarCopy.save()

        hydrogenPEMOnshoreWindCopy = hydrogenPEMGridActCopy.copy(name = 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from onshore wind electricity', location = newLoc)
        elecHydrogenPEMOnshoreWindCopy = [exchange for exchange in hydrogenPEMOnshoreWindCopy.technosphere()
                                          if 'electricity, low voltage' in exchange['product']][0]
        elecHydrogenPEMOnshoreWindCopy.input = onshoreWindElecAct[0]
        elecHydrogenPEMOnshoreWindCopy.save()

        hydrogenPEMOffshoreWindCopy = hydrogenPEMGridActCopy.copy(name = 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from offshore wind electricity', location = newLoc)
        elecHydrogenPEMOffshoreWindCopy = [exchange for exchange in hydrogenPEMOffshoreWindCopy.technosphere()
                                          if 'electricity, low voltage' in exchange['product']][0]
        elecHydrogenPEMOffshoreWindCopy.input = offshoreWindElecAct[0]
        elecHydrogenPEMOffshoreWindCopy.save()

        hydrogenPEMNuclearCopy = hydrogenPEMGridActCopy.copy(name = 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from nuclear electricity', location = newLoc)
        elecHydrogenPEMNuclearCopy = [exchange for exchange in hydrogenPEMNuclearCopy.technosphere()
                                          if 'electricity, low voltage' in exchange['product']][0]
        elecHydrogenPEMNuclearCopy.input = nuclearElecAct[0]
        elecHydrogenPEMNuclearCopy.save()

        hydrogenPEMHydroCopy = hydrogenPEMGridActCopy.copy(name = 'hydrogen production, gaseous, 30 bar, from PEM electrolysis, from hydro electricity', location = newLoc)
        elecHydrogenPEMHydroCopy = [exchange for exchange in hydrogenPEMHydroCopy.technosphere()
                                          if 'electricity, low voltage' in exchange['product']][0]
        elecHydrogenPEMHydroCopy.input = hydroElecAct[0]
        elecHydrogenPEMHydroCopy.save()
        
        # ammonia inventories
        ammoniaPEMGridAct = [activity for activity in mySSP2DB 
                          if 'ammonia; hydrogen, PEM electrolysis, electricity mix' in activity['name']
                          and newLoc in activity['location']][0]
        ammoniaPEMSolarAct = ammoniaPEMGridAct.copy(name = ammoniaPEMGridAct['name'].replace('mix', 'solar'))
        ammoniaPEMOnshoreWindAct = ammoniaPEMGridAct.copy(name = ammoniaPEMGridAct['name'].replace('mix', 'onshore wind'))
        ammoniaPEMOffshoreWindAct = ammoniaPEMGridAct.copy(name = ammoniaPEMGridAct['name'].replace('mix', 'offshore wind'))
        ammoniaPEMNuclearAct = ammoniaPEMGridAct.copy(name = ammoniaPEMGridAct['name'].replace('mix', 'nuclear'))
        ammoniaPEMHydroAct = ammoniaPEMGridAct.copy(name = ammoniaPEMGridAct['name'].replace('mix', 'hydro'))

        hydrogenGridExchange = ammoniaPEMGridAct.new_exchange(input = hydrogenPEMGridActCopy, amount = 0.176, type = 'technosphere')
        hydrogenGridExchange.save()
        hydrogenSolarExchange = ammoniaPEMSolarAct.new_exchange(input = hydrogenPEMSolarCopy, amount = 0.176, type = 'technosphere')
        hydrogenSolarExchange.save()
        hydrogenOnshoreWindExchange = ammoniaPEMOnshoreWindAct.new_exchange(input = hydrogenPEMOnshoreWindCopy, amount = 0.176, type = 'technosphere')
        hydrogenOnshoreWindExchange.save()
        hydrogenOffshoreWindExchange = ammoniaPEMOffshoreWindAct.new_exchange(input = hydrogenPEMOffshoreWindCopy, amount = 0.176, type = 'technosphere')
        hydrogenOffshoreWindExchange.save()
        hydrogenNuclearExchange = ammoniaPEMNuclearAct.new_exchange(input = hydrogenPEMNuclearCopy, amount = 0.176, type = 'technosphere')
        hydrogenNuclearExchange.save()
        hydrogenHydroExchange = ammoniaPEMHydroAct.new_exchange(input = hydrogenPEMHydroCopy, amount = 0.176, type = 'technosphere')
        hydrogenHydroExchange.save()

        # carbon dioxide inventories
        carbonDioxideSolventActivites = [activity for activity in ecoinventSSP2DB 
                    if 'carbon dioxide, captured from atmosphere, with a solvent-based' in activity['name']
                    and location == activity['location']]
        for carbonDioxideSolventActivity in carbonDioxideSolventActivites:
            carbonDioxideSolventGridCopy = carbonDioxideSolventActivity.copy(database = mySSP2DB.name, location = newLoc)
            carbonDioxideSolventGridCopy.update({'reference product' : 'carbon dioxide, Abhi'})
            carbonDioxideSolventGridCopy.save()

            carbonDioxideSolventOnshoreWindCopy = carbonDioxideSolventGridCopy.copy(name = carbonDioxideSolventGridCopy['name'].replace('grid', 'onshore wind'), location = newLoc)
            elecOnshoreWindCopy = [exchange for exchange in carbonDioxideSolventOnshoreWindCopy.technosphere()
                                        if 'electricity, medium voltage' in exchange['product']][0]
            elecOnshoreWindCopy.input = onshoreWindElecAct[0]
            elecOnshoreWindCopy.save()

        carbonDioxideSorbentActivites = [activity for activity in ecoinventSSP2DB 
                    if 'carbon dioxide, captured from atmosphere, with a sorbent-based' in activity['name']
                    and location == activity['location']]
        for carbonDioxideSorbentActivity in carbonDioxideSorbentActivites:
            carbonDioxideSorbentGridCopy = carbonDioxideSorbentActivity.copy(database = mySSP2DB.name, location = newLoc)
            carbonDioxideSorbentGridCopy.update({'reference product' : 'carbon dioxide, Abhi'})
            carbonDioxideSorbentGridCopy.save()

            carbonDioxideSorbentOnshoreWindCopy = carbonDioxideSorbentGridCopy.copy(name = carbonDioxideSorbentGridCopy['name'].replace('grid', 'onshore wind'), location = newLoc)
            elecOnshoreWindCopy = [exchange for exchange in carbonDioxideSorbentOnshoreWindCopy.technosphere()
                                        if 'electricity, medium voltage' in exchange['product']][0]
            elecOnshoreWindCopy.input = onshoreWindElecAct[0]
            elecOnshoreWindCopy.save()

            carbonDioxideSorbentGeothermalCopy = carbonDioxideSorbentGridCopy.copy(name = carbonDioxideSorbentGridCopy['name'].replace('grid', 'geothermal'), location = newLoc)
            elecGeothermalCopy = [exchange for exchange in carbonDioxideSorbentGeothermalCopy.technosphere()
                                        if 'electricity, medium voltage' in exchange['product']][0]
            elecGeothermalCopy.input = geothermalElecAct[0]
            elecGeothermalCopy.save()
            
    print('Finished ', mySSP2DBName)

Started lci-Abhi image SSP2-Base 2020
Finished  lci-Abhi image SSP2-Base 2020
Started lci-Abhi image SSP2-Base 2030
Finished  lci-Abhi image SSP2-Base 2030
Started lci-Abhi image SSP2-Base 2040
Finished  lci-Abhi image SSP2-Base 2040
Started lci-Abhi image SSP2-Base 2050
Finished  lci-Abhi image SSP2-Base 2050
Started lci-Abhi image SSP2-RCP19 2030
Finished  lci-Abhi image SSP2-RCP19 2030
Started lci-Abhi image SSP2-RCP19 2040
Finished  lci-Abhi image SSP2-RCP19 2040
Started lci-Abhi image SSP2-RCP19 2050
Finished  lci-Abhi image SSP2-RCP19 2050
Started lci-Abhi image SSP2-RCP26 2030
Finished  lci-Abhi image SSP2-RCP26 2030
Started lci-Abhi image SSP2-RCP26 2040
Finished  lci-Abhi image SSP2-RCP26 2040
Started lci-Abhi image SSP2-RCP26 2050
Finished  lci-Abhi image SSP2-RCP26 2050


In [11]:
# generate all methanol inventories
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database(ecoinventSSP2DBName)
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database(mySSP2DBName)
    print('Started ' + mySSP2DBName)

    locations = imageLocations

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
        if ecoinventLocation:
            newLoc = ecoinventLocation    
        else:
            newLoc = 'GLO'

        # extract hydrogen inventories for a particular location from my database
        hydrogenInventories = [activity for activity in mySSP2DB
                               if 'hydrogen, Abhi' in activity['reference product']
                               and newLoc in activity['location']]
        
        # extract carbon dioxide inventories for a particular location from my database
        carbonDioxideInventories = [activity for activity in mySSP2DB
                                    if 'carbon dioxide, Abhi' in activity['reference product']
                                    and newLoc in activity['location']]
        # methanol inventories
        greenMethanolGridAct = [activity for activity in ecoinventSSP2DB
                                if 'methanol synthesis, hydrogen from electrolysis, CO2 from DAC' in activity['name']
                                and location in activity['location']][0]
        
        for carbonDioxideInventory in carbonDioxideInventories:
            for hydrogenInventory in hydrogenInventories:
                greenMethanolGridActCopy = greenMethanolGridAct.copy(name = 'methanol synthesis, ' + 
                                                                     hydrogenInventory['name'] + ' '+
                                                                     carbonDioxideInventory['name'], database = mySSP2DB.name, location = newLoc)
                greenMethanolGridActCopy.update({'reference product' : 'methanol, Abhi'})
                greenMethanolGridActCopy.save()
                hydrogenExchange = [exchange for exchange in greenMethanolGridActCopy.technosphere() if 'hydrogen' in exchange['name']][0]
                carbonDioxideExchange = [exchange for exchange in greenMethanolGridActCopy.technosphere() if 'carbon dioxide' in exchange['name']][0]
                hydrogenExchange.input = hydrogenInventory
                hydrogenExchange.save()
                carbonDioxideExchange.input = carbonDioxideInventory
                carbonDioxideExchange.save()
  
    print('Finished ', mySSP2DBName)

Started lci-Abhi image SSP2-Base 2020
Finished  lci-Abhi image SSP2-Base 2020
Started lci-Abhi image SSP2-Base 2030
Finished  lci-Abhi image SSP2-Base 2030
Started lci-Abhi image SSP2-Base 2040
Finished  lci-Abhi image SSP2-Base 2040
Started lci-Abhi image SSP2-Base 2050
Finished  lci-Abhi image SSP2-Base 2050
Started lci-Abhi image SSP2-RCP19 2030
Finished  lci-Abhi image SSP2-RCP19 2030
Started lci-Abhi image SSP2-RCP19 2040
Finished  lci-Abhi image SSP2-RCP19 2040
Started lci-Abhi image SSP2-RCP19 2050
Finished  lci-Abhi image SSP2-RCP19 2050
Started lci-Abhi image SSP2-RCP26 2030
Finished  lci-Abhi image SSP2-RCP26 2030
Started lci-Abhi image SSP2-RCP26 2040
Finished  lci-Abhi image SSP2-RCP26 2040
Started lci-Abhi image SSP2-RCP26 2050
Finished  lci-Abhi image SSP2-RCP26 2050


In [12]:
# generate all ethylene inventories
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database(mySSP2DBName)
    print('Started ' + mySSP2DBName)

    locations = imageLocations

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
        if ecoinventLocation:
            newLoc = ecoinventLocation    
        else:
            newLoc = 'GLO'

        # ethylene inventories
        ethyleneInventory = [activity for activity in mySSP2DB
                                if 'ethylene, Abhi' in activity['reference product']
                                and newLoc in activity['location']][0]
        
        methanolInventories = [activity for activity in mySSP2DB
                               if 'methanol, Abhi' in activity['reference product']
                               and 'methanol, BAU' not in activity['name']
                               and newLoc in activity['location']]
        
        for methanolInventory in methanolInventories:

            ethyleneInventoryCopy = ethyleneInventory.copy(name = 'ethylene, MTO; ' + methanolInventory['name'], location = newLoc)
            ethyleneInventoryCopy.update({'reference product' : 'ethylene, Abhi'})
            ethyleneInventoryCopy.save()

            methanolExchange = [exchange for exchange in ethyleneInventoryCopy.technosphere()
                                 if 'methanol, BAU' in exchange['name']][0]
            methanolExchange.input = methanolInventory
            methanolExchange.save()
        
  
    print('Finished ', mySSP2DBName)

Started lci-Abhi image SSP2-Base 2020
Finished  lci-Abhi image SSP2-Base 2020
Started lci-Abhi image SSP2-Base 2030
Finished  lci-Abhi image SSP2-Base 2030
Started lci-Abhi image SSP2-Base 2040
Finished  lci-Abhi image SSP2-Base 2040
Started lci-Abhi image SSP2-Base 2050
Finished  lci-Abhi image SSP2-Base 2050
Started lci-Abhi image SSP2-RCP19 2030
Finished  lci-Abhi image SSP2-RCP19 2030
Started lci-Abhi image SSP2-RCP19 2040
Finished  lci-Abhi image SSP2-RCP19 2040
Started lci-Abhi image SSP2-RCP19 2050
Finished  lci-Abhi image SSP2-RCP19 2050
Started lci-Abhi image SSP2-RCP26 2030
Finished  lci-Abhi image SSP2-RCP26 2030
Started lci-Abhi image SSP2-RCP26 2040
Finished  lci-Abhi image SSP2-RCP26 2040
Started lci-Abhi image SSP2-RCP26 2050
Finished  lci-Abhi image SSP2-RCP26 2050


In [13]:
# generate all hydrogen inventories without electricity
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database(ecoinventSSP2DBName)
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database(mySSP2DBName)
    print('Started ' + mySSP2DBName)

    locations = imageLocations

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
        if ecoinventLocation:
            newLoc = ecoinventLocation    
        else:
            newLoc = 'GLO'

         # hydrogen inventories
        hydrogenBAUAct = [activity for activity in ecoinventSSP2DB 
                    if 'hydrogen production, steam methane reforming of natural gas, 25 bar' in activity['name']
                    and location in activity['location']][0]
        hydrogenBAUActCopy = hydrogenBAUAct.copy(database = mySSP2DB.name, location = newLoc, name = 'hydrogen production without electricity, steam reforming of natural gas, 25 bar')
        hydrogenBAUActCopy.update({'reference product' : 'hydrogen, Abhi'})
        hydrogenBAUActCopy.save()
        elecExchange = [exchange for exchange in hydrogenBAUActCopy.technosphere()
                                        if 'electricity, high voltage' in exchange['product']][0]
        elecExchange.update({'amount' : 0})
        elecExchange.save()
        
        
        hydrogenBlueAct = [activity for activity in ecoinventSSP2DB 
                       if r'hydrogen production, steam methane reforming of natural gas, with CCS (MDEA, 98% eff.), 25 bar' in activity['name']
                       and location in activity['location']][0]
        hydrogenBlueActCopy = hydrogenBlueAct.copy(database = mySSP2DB.name, location = newLoc, name = r'hydrogen production without electricity, steam methane reforming of natural gas, with CCS (MDEA, 98% eff.), 25 bar')
        hydrogenBlueActCopy.update({'reference product' : 'hydrogen, Abhi'})
        hydrogenBlueActCopy.save()    
        elecExchange = [exchange for exchange in hydrogenBlueActCopy.technosphere()
                                        if 'electricity, high voltage' in exchange['product']][0]
        elecExchange.update({'amount' : 0})
        elecExchange.save()

    print('Finished ', mySSP2DBName)

Started lci-Abhi image SSP2-Base 2020
Finished  lci-Abhi image SSP2-Base 2020
Started lci-Abhi image SSP2-Base 2030
Finished  lci-Abhi image SSP2-Base 2030
Started lci-Abhi image SSP2-Base 2040
Finished  lci-Abhi image SSP2-Base 2040
Started lci-Abhi image SSP2-Base 2050
Finished  lci-Abhi image SSP2-Base 2050
Started lci-Abhi image SSP2-RCP19 2030
Finished  lci-Abhi image SSP2-RCP19 2030
Started lci-Abhi image SSP2-RCP19 2040
Finished  lci-Abhi image SSP2-RCP19 2040
Started lci-Abhi image SSP2-RCP19 2050
Finished  lci-Abhi image SSP2-RCP19 2050
Started lci-Abhi image SSP2-RCP26 2030
Finished  lci-Abhi image SSP2-RCP26 2030
Started lci-Abhi image SSP2-RCP26 2040
Finished  lci-Abhi image SSP2-RCP26 2040
Started lci-Abhi image SSP2-RCP26 2050
Finished  lci-Abhi image SSP2-RCP26 2050


In [14]:
endTime = time.time() # end time
elapsedTime = endTime - startTime # calculate elapsed time
print(f'Elapsed time: {elapsedTime/3600:.2f} hours') 

Elapsed time: 8.44 hours
